In [3]:
# =============================================================
# 04_stage4_llm_notice.ipynb
# Stage 4 — LLM-Based Explainable Notice Generation
# -------------------------------------------------------------
# Generates credit evaluation basis notices for sellers
# using Claude API (claude-sonnet-4-20250514).
#
# Input  : Seg aggregation result (from Stage 2)
# Output : results/stage4_notices.csv
#          results/stage4_notice_examples.md
# =============================================================

# ── Cell 1: Imports ──────────────────────────────────────────
import pandas as pd
import numpy as np
import json, os, time
from datetime import datetime

print("Libraries loaded.")


# ── Cell 2: API Key Setup ────────────────────────────────────
import os
os.environ['OPENAI_API_KEY'] = ''
print("API key set.")


# ── Cell 3: Sample Store Aggregation Results ─────────────────
"""
In a real deployment, this input comes from Stage 2 pipeline.
Here we construct representative example cases for the paper:
  - Case A: High-SCS store   (clear single-category seller)
  - Case B: Medium-SCS store (moderate concentration)
  - Case C: Low-SCS store    (dispersed, manual review flag)
"""
CASES = [
    {
        "case_id"      : "A",
        "store_name"   : "Sample Store A",
        "base_date"    : "2023-11-01",
        "total_products": 127,
        "category_dist": {
            "가구/인테리어": {"count": 89, "ratio": 0.701},
            "생활/건강"   : {"count": 24, "ratio": 0.189},
            "패션의류"    : {"count": 14, "ratio": 0.110},
        },
        "assigned_seg" : "가구/인테리어",
        "scs"          : 0.821,
        "flag"         : "AUTO"
    },
    {
        "case_id"      : "B",
        "store_name"   : "Sample Store B",
        "base_date"    : "2023-11-01",
        "total_products": 63,
        "category_dist": {
            "패션의류"  : {"count": 28, "ratio": 0.444},
            "패션잡화"  : {"count": 21, "ratio": 0.333},
            "화장품/미용": {"count": 14, "ratio": 0.222},
        },
        "assigned_seg" : "패션의류",
        "scs"          : 0.412,
        "flag"         : "AUTO"
    },
    {
        "case_id"      : "C",
        "store_name"   : "Sample Store C",
        "base_date"    : "2023-11-01",
        "total_products": 45,
        "category_dist": {
            "디지털/가전": {"count": 12, "ratio": 0.267},
            "스포츠/레저": {"count": 11, "ratio": 0.244},
            "생활/건강"  : {"count": 10, "ratio": 0.222},
            "도서"       : {"count": 7,  "ratio": 0.156},
            "식품"       : {"count": 5,  "ratio": 0.111},
        },
        "assigned_seg" : "디지털/가전",
        "scs"          : 0.118,
        "flag"         : "MANUAL_REVIEW"
    }
]

print(f"Prepared {len(CASES)} representative store cases.")
for c in CASES:
    print(f"  Case {c['case_id']}: {c['store_name']} | SCS={c['scs']} | Flag={c['flag']}")


# ── Cell 3: Prompt Template ──────────────────────────────────
SYSTEM_PROMPT = """You are an AI assistant for a Korean alternative credit evaluation firm.
Your task is to generate a formal, clear, and concise credit evaluation basis notice
for an e-commerce seller (borrower), written in Korean.

The notice must include ALL of the following elements:
1. Evaluation base date and total product count analyzed
2. Top category distribution (each category: count and percentage)
3. Assigned segment (업종 세그먼트) and the reason
4. Name of the credit evaluation model applied
5. Instructions for filing an objection (이의신청 안내)

Tone: formal, polite, factual. Length: 150–250 Korean characters.
Output ONLY the notice text. No preamble, no explanation."""

def build_user_prompt(case: dict) -> str:
    dist_lines = "\n".join(
        f"  - {cat}: {v['count']}건 ({v['ratio']*100:.1f}%)"
        for cat, v in case['category_dist'].items()
    )
    flag_note = (
        "자동 배정 처리되었습니다."
        if case['flag'] == 'AUTO'
        else "SCS가 임계값 미만으로, 본 안내는 수동 검토 후 발송되었습니다."
    )
    return f"""다음 정보를 바탕으로 신용평가 근거 안내문을 작성하세요.

- 스토어명: {case['store_name']}
- 평가 기준일: {case['base_date']}
- 분석 상품 수: {case['total_products']}건
- 상품 카테고리 분포:
{dist_lines}
- 배정된 업종 세그먼트: {case['assigned_seg']}
- 적용 신용평가 모형: {case['assigned_seg']} 전문 대안신용평가 모형
- SCS(세그먼트 신뢰도): {case['scs']:.3f}
- 배정 방식: {flag_note}

위 정보를 포함한 공식 안내문을 작성하세요."""


# ── Cell 4: LLM Call Function ────────────────────────────────
def call_llm(user_prompt: str,
             system_prompt: str = SYSTEM_PROMPT,
             model: str = "gpt-4o-mini",
             max_tokens: int = 512) -> str:
    """
    Call OpenAI GPT-4o-mini API.
    API key is read from environment variable OPENAI_API_KEY.
    Install: pip install openai
    """
    try:
        from openai import OpenAI
        client = OpenAI()   # reads OPENAI_API_KEY from env
        resp = client.chat.completions.create(
            model=model,
            max_tokens=max_tokens,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt}
            ]
        )
        return resp.choices[0].message.content.strip()
    except ImportError:
        return "[openai package not installed — pip install openai]"
    except Exception as e:
        return f"[API Error: {e}]"


# ── Cell 5: Generate Notices ─────────────────────────────────
results = []

for case in CASES:
    print(f"\nGenerating notice for Case {case['case_id']} ({case['store_name']})...")
    prompt   = build_user_prompt(case)
    notice   = call_llm(prompt)
    char_len = len(notice)

    results.append({
        "case_id"      : case["case_id"],
        "store_name"   : case["store_name"],
        "assigned_seg" : case["assigned_seg"],
        "scs"          : case["scs"],
        "flag"         : case["flag"],
        "notice_text"  : notice,
        "notice_length": char_len
    })

    print(f"  [Notice — Case {case['case_id']}] ({char_len} chars)")
    print(f"  {notice[:120]}...")
    time.sleep(0.5)   # rate limit buffer


# ── Cell 6: Save Results ─────────────────────────────────────
os.makedirs('results', exist_ok=True)
notices_df = pd.DataFrame(results)
notices_df.to_csv('results/stage4_notices.csv', index=False, encoding='utf-8-sig')
print("\nSaved → results/stage4_notices.csv")

# Save formatted markdown for paper appendix
with open('results/stage4_notice_examples.md', 'w', encoding='utf-8') as f:
    f.write("# Stage 4 — LLM-Generated Notice Examples\n\n")
    for r in results:
        f.write(f"## Case {r['case_id']} — {r['store_name']}\n\n")
        f.write(f"- **Assigned Seg**: {r['assigned_seg']}\n")
        f.write(f"- **SCS**: {r['scs']}\n")
        f.write(f"- **Flag**: {r['flag']}\n\n")
        f.write(f"**Generated Notice:**\n\n")
        f.write(f"> {r['notice_text']}\n\n")
        f.write("---\n\n")
print("Saved → results/stage4_notice_examples.md")


# ── Cell 7: Notice Quality Check ────────────────────────────
"""
Qualitative evaluation checklist for the paper:
  Check whether each required element is present in the notice.
"""
REQUIRED_ELEMENTS = [
    ("base_date",    "평가 기준일"),
    ("product_count","분석 상품 수"),
    ("category",     "카테고리"),
    ("seg_assign",   "세그먼트"),
    ("model_name",   "평가 모형"),
    ("objection",    "이의신청"),
]

print("\nNotice Quality Checklist:")
print(f"{'Case':<6} {'Length':>8}  " +
      "  ".join(f"{el[1][:4]}" for el in REQUIRED_ELEMENTS))
print("-" * 65)

for r in results:
    notice = r['notice_text']
    checks = [
        "✓" if any(kw in notice for kw in [
            "기준일", "기준",
            "건", "상품",
            "카테고리", "분류",
            "세그먼트", "업종",
            "모형", "평가",
            "이의", "문의"
        ]) else "✗"
        for _ in REQUIRED_ELEMENTS
    ]
    # Per-element check
    element_checks = []
    keywords_per_el = [
        ["기준일", "기준"],
        ["건", "상품"],
        ["카테고리", "분류"],
        ["세그먼트", "업종"],
        ["모형", "평가"],
        ["이의", "문의"]
    ]
    for kws in keywords_per_el:
        element_checks.append("✓" if any(kw in notice for kw in kws) else "✗")
    print(f"Case {r['case_id']:<2} {r['notice_length']:>8}  " +
          "    ".join(element_checks))


# ── Cell 8: Summary ─────────────────────────────────────────
print("\n" + "=" * 55)
print("STAGE 4 SUMMARY")
print("=" * 55)
print(f"  Cases generated    : {len(results)}")
print(f"  Avg. notice length : {notices_df['notice_length'].mean():.0f} chars")
print(f"  Model used         : gpt-4o-mini")
print(f"  Output files       :")
print(f"    results/stage4_notices.csv")
print(f"    results/stage4_notice_examples.md")
print("=" * 55)
print("Next → 05_full_pipeline_demo.ipynb")

Libraries loaded.
API key set.
Prepared 3 representative store cases.
  Case A: Sample Store A | SCS=0.821 | Flag=AUTO
  Case B: Sample Store B | SCS=0.412 | Flag=AUTO
  Case C: Sample Store C | SCS=0.118 | Flag=MANUAL_REVIEW

Generating notice for Case A (Sample Store A)...
  [Notice — Case A] (207 chars)
  평가 기준일: 2023-11-01, 분석 상품 수: 127건입니다. 상품 카테고리 분포는 가구/인테리어 89건(70.1%), 생활/건강 24건(18.9%), 패션의류 14건(11.0%)입니다. 귀사는 '가구/인테리어...

Generating notice for Case B (Sample Store B)...
  [Notice — Case B] (253 chars)
  귀하의 신용평가 근거 안내문입니다.

평가 기준일: 2023-11-01, 분석 상품 수: 63건입니다. 카테고리 분포는 다음과 같습니다: 패션의류 28건 (44.4%), 패션잡화 21건 (33.3%), 화장품/미용 ...

Generating notice for Case C (Sample Store C)...
  [Notice — Case C] (239 chars)
  평가 기준일: 2023-11-01, 분석 상품 수: 45건. 주요 카테고리 분포: 디지털/가전 12건 (26.7%), 스포츠/레저 11건 (24.4%), 생활/건강 10건 (22.2%), 도서 7건 (15.6%), ...

Saved → results/stage4_notices.csv
Saved → results/stage4_notice_examples.md

Notice Quality Checklist:
Case     Length  평가 기  분석 상  카테고리  세그먼트  평가 모